In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/plant_training

plant_dataset_clean.zip


In [ ]:
!unzip -q "/content/drive/MyDrive/plant_training/plant_dataset_clean.zip" -d /content/dataset

In [ ]:
!ls -F /content/dataset

test/  train/


In [ ]:
#creating val folder
import os
import shutil
import random

# Define paths
train_dir = '/content/dataset/train'
val_dir = '/content/dataset/val'

# Create the val directory if it doesn't exist
os.makedirs(val_dir, exist_ok=True)

# Get list of classes (subfolders in train)
classes = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]

for cls in classes:
    # Create class subfolder in val
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)

    # Get all images for this class
    src_path = os.path.join(train_dir, cls)
    images = os.listdir(src_path)

    # Shuffle and pick 20% to move
    random.shuffle(images)
    num_val = int(len(images) * 0.2)
    val_images = images[:num_val]

    # Move the files
    for img in val_images:
        shutil.move(os.path.join(src_path, img), os.path.join(val_dir, cls, img))

print(f"Split complete. 'val' folder created with {len(classes)} classes.")

Split complete. 'val' folder created with 28 classes.


In [ ]:
base_path = '/content/dataset'

# Check every class in train
for cls in os.listdir(os.path.join(base_path, 'train')):
    train_cls_ptr = os.path.join(base_path, 'train', cls)
    val_cls_ptr = os.path.join(base_path, 'val', cls)

    if os.path.isdir(train_cls_ptr):
        train_files = os.listdir(train_cls_ptr)
        val_files = os.listdir(val_cls_ptr) if os.path.exists(val_cls_ptr) else []

        # If train is empty, copy one image FROM val TO train
        if len(train_files) == 0 and len(val_files) > 0:
            shutil.copy(os.path.join(val_cls_ptr, val_files[0]), os.path.join(train_cls_ptr, val_files[0]))
            print(f"Fixed: Copied image to train/{cls} (original stayed in val)")

        # If val is empty, copy one image FROM train TO val
        elif len(val_files) == 0 and len(train_files) > 0:
            os.makedirs(val_cls_ptr, exist_ok=True)
            shutil.copy(os.path.join(train_cls_ptr, train_files[0]), os.path.join(val_cls_ptr, train_files[0]))
            print(f"Fixed: Copied image to val/{cls} (original stayed in train)")

In [ ]:
# Check if the folder is empty or has weird files
!ls -a "/content/dataset/train/Tomato_two_spotted_spider_mites_leaf"
!ls -a "/content/dataset/val/Tomato_two_spotted_spider_mites_leaf"

.  ..  plantdoc_train_Tomato_two_spotted_spider_mites_leaf_00001.jpg
.  ..  plantdoc_train_Tomato_two_spotted_spider_mites_leaf_00001.jpg


In [15]:
import torch
from torchvision import datasets, transforms

# Define the device (Crucial fix for your NameError)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Data transformations for ResNet-50
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = '/content/dataset'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val']}

# Using batch_size=16 for ResNet-50 to prevent memory errors
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=16,
                                             shuffle=True, num_workers=2)
              for x in ['train', 'val']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

Using device: cuda:0


In [16]:
import torch.nn as nn
from torchvision import models

# Load ResNet-50
model_ft = models.resnet50(weights='DEFAULT')

# ResNet-50 uses 2048 input features for the final layer
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, 28)

# This will now work without the NameError
model_ft = model_ft.to(device)

In [18]:
import time
import copy

def train_model(model, criterion, optimizer, scheduler, num_epochs=15):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Deep copy the model if it has the best validation accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [19]:
# Start training
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=15)

# Save for Milestone 3 Integration
torch.save(model_ft.state_dict(), '/content/drive/MyDrive/plant_disease_resnet50_v1.pth')
print("ResNet-50 Training Complete and Saved to Drive.")

Epoch 0/14
----------
train Loss: 1.1436 Acc: 0.6938
val Loss: 0.3327 Acc: 0.9088

Epoch 1/14
----------
train Loss: 0.3803 Acc: 0.8858
val Loss: 0.3399 Acc: 0.9348

Epoch 2/14
----------
train Loss: 0.2962 Acc: 0.9095
val Loss: 0.2121 Acc: 0.9438

Epoch 3/14
----------
train Loss: 0.2429 Acc: 0.9258
val Loss: 0.2019 Acc: 0.9492

Epoch 4/14
----------
train Loss: 0.2125 Acc: 0.9333
val Loss: 0.1582 Acc: 0.9554

Epoch 5/14
----------
train Loss: 0.1936 Acc: 0.9409
val Loss: 0.1228 Acc: 0.9607

Epoch 6/14
----------
train Loss: 0.1761 Acc: 0.9443
val Loss: 0.1526 Acc: 0.9582

Epoch 7/14
----------
train Loss: 0.1517 Acc: 0.9533
val Loss: 0.2106 Acc: 0.9588

Epoch 8/14
----------
train Loss: 0.1411 Acc: 0.9567
val Loss: 0.1684 Acc: 0.9601

Epoch 9/14
----------
train Loss: 0.1464 Acc: 0.9551
val Loss: 0.1774 Acc: 0.9605

Epoch 10/14
----------
train Loss: 0.1391 Acc: 0.9565
val Loss: 0.1931 Acc: 0.9611

Epoch 11/14
----------
train Loss: 0.1364 Acc: 0.9573
val Loss: 0.1757 Acc: 0.9617

Ep

In [22]:
import torch

def check_manual_accuracy(model, dataloader):
    model.eval()  # Set to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # Calculate percentage and format to 2 digits
    accuracy = 100 * correct / total
    print(f'Manual Validation Accuracy: {accuracy:.2f}%')
    return accuracy

# Run it
manual_acc = check_manual_accuracy(model_ft, dataloaders['val'])

Manual Validation Accuracy: 96.50%


In [23]:
import os
import random

# 1. Automatically find a valid image path from your unzipped dataset
val_dir = '/content/dataset/val' # Change this if your unzipped folder name is different
random_class = random.choice(os.listdir(val_dir))
random_image = random.choice(os.listdir(os.path.join(val_dir, random_class)))
actual_test_path = os.path.join(val_dir, random_class, random_image)

# 2. Run the prediction
print(f"Testing on: {actual_test_path}")
predict_plant_disease(actual_test_path, model_ft, class_names)

Testing on: /content/dataset/val/Tomato_two_spotted_spider_mites_leaf/plantdoc_train_Tomato_two_spotted_spider_mites_leaf_00001.jpg
Predicted Disease: Tomato___Bacterial_spot
Confidence Score: 43.41%


In [24]:
def check_manual_accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # Calculation for two-digit display
    raw_acc = (correct / total) * 100
    print("-" * 30)
    print(f"TOTAL IMAGES: {total}")
    print(f"CORRECT PREDICTIONS: {correct}")
    print(f"FINAL ACCURACY: {raw_acc:.2f}%") # This gives you the 2-digit format (e.g., 96.45%)
    print("-" * 30)

# Execute the check
check_manual_accuracy(model_ft, dataloaders['val'])

------------------------------
TOTAL IMAGES: 5140
CORRECT PREDICTIONS: 4960
FINAL ACCURACY: 96.50%
------------------------------


In [26]:
import shutil

# Copying the ResNet model
shutil.copy('/content/plant_disease_resnet50_v1.pth', '/content/drive/plant_disease_resnet50_v1.pth')


FileNotFoundError: [Errno 2] No such file or directory: '/content/plant_disease_resnet50_v1.pth'